# TF-IDF + Logistic Regression Baseline

**Task:** Binary text classification (e.g. sentiment / stance detection)  
**Languages:** English (`english.xlsx`) · Tagalog (`tagalog.xlsx`)  
**Model:** Bigram TF-IDF → Logistic Regression pipeline  

## Experimental Conditions

| Condition | Train | Evaluated on |
|-----------|-------|--------------|
| English-only | English | English (in-domain), Tagalog (cross-lingual zero-shot), Combined |
| Tagalog-only | Tagalog | Tagalog (in-domain), English (cross-lingual zero-shot), Combined |
| Bilingual | English + Tagalog (combined) | Combined, English subset, Tagalog subset |

All CV results use **10-fold Stratified K-Fold** with `random_state=42`.

## Reproducibility

- All random seeds are fixed to `42`.
- Install dependencies: `pip install -r requirements_baseline.txt`
- Place `english.xlsx` and `tagalog.xlsx` in the repo root (or update `DATA_DIR` below).

---
## 0 · Setup

In [ ]:
!pip install -q pandas openpyxl scikit-learn numpy

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline

DATA_DIR    = "."          # Directory containing english.xlsx and tagalog.xlsx
RESULTS_DIR = "results"    # Output directory for CSVs
os.makedirs(RESULTS_DIR, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE    = 0.10    # 90/10 stratified split
N_SPLITS     = 10      # Stratified K-Fold folds

print("Setup complete.")

---
## 1 · Data Loading & Validation

In [ ]:
def load_xlsx(path: str) -> pd.DataFrame:
    """
    Load a two-column Excel file (text | label) with no header.

    Applies:
    - Strip whitespace from text
    - Coerce labels to int; drop rows where label is unparseable
    - Drop rows with empty text

    Parameters
    ----------
    path : str
        Path to the .xlsx file.

    Returns
    -------
    pd.DataFrame with columns ['text', 'label']
    """
    raw = pd.read_excel(path, header=None)
    df  = pd.DataFrame({
        "text":  raw.iloc[:, 0].fillna("").astype(str).str.strip(),
        "label": pd.to_numeric(raw.iloc[:, 1], errors="coerce"),
    })
    df = df.dropna(subset=["label"]).copy()
    df["label"] = df["label"].astype(int)
    df = df[df["text"].str.len() > 0].reset_index(drop=True)
    return df


df_en = load_xlsx(os.path.join(DATA_DIR, "english.xlsx"))
df_tl = load_xlsx(os.path.join(DATA_DIR, "tagalog.xlsx"))

for name, df in [("English", df_en), ("Tagalog", df_tl)]:
    print(f"\n{name}  shape: {df.shape}")
    print(df["label"].value_counts().sort_index().to_string())

---
## 2 · Train / Test Split

In [ ]:
def make_splits(
    df: pd.DataFrame,
    test_size: float = TEST_SIZE,
    random_state: int = RANDOM_STATE,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Stratified train/test split.

    Returns
    -------
    (train_df, test_df) — both reset-indexed.
    """
    train, test = train_test_split(
        df, test_size=test_size, stratify=df["label"], random_state=random_state
    )
    return train.reset_index(drop=True), test.reset_index(drop=True)


en_train, en_test = make_splits(df_en)
tl_train, tl_test = make_splits(df_tl)

print(f"English  — train: {en_train.shape}  test: {en_test.shape}")
print(f"Tagalog  — train: {tl_train.shape}  test: {tl_test.shape}")

---
## 3 · Multilingual (Combined) Dataset

In [ ]:
def combine_lang_dfs(
    df_a: pd.DataFrame,
    df_b: pd.DataFrame,
    label_a: str,
    label_b: str,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    """Concatenate two language DataFrames, add a 'lang' column, and shuffle."""
    return (
        pd.concat([
            df_a.assign(lang=label_a),
            df_b.assign(lang=label_b),
        ], ignore_index=True)
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )


df_all  = combine_lang_dfs(df_en,    df_tl,    "en", "tl")   # full combined
train_df = combine_lang_dfs(en_train, tl_train, "en", "tl")   # combined train
test_df  = combine_lang_dfs(en_test,  tl_test,  "en", "tl")   # combined test

for name, df in [("Full combined", df_all), ("Combined train", train_df), ("Combined test", test_df)]:
    print(f"\n{name}  shape: {df.shape}")
    print(df["lang"].value_counts().to_string())

---
## 4 · Model Pipeline

In [ ]:
def build_pipeline(random_state: int = RANDOM_STATE) -> Pipeline:
    """
    Build a TF-IDF → Logistic Regression pipeline.

    TF-IDF configuration
    --------------------
    - Unigrams + bigrams  (ngram_range=(1, 2))
    - Vocabulary capped at 20,000 features
    - min_df=2 to filter very rare tokens
    - max_df=0.95 to filter near-universal tokens
    - Sublinear TF scaling  (1 + log(tf))

    Logistic Regression configuration
    -----------------------------------
    - Solver: liblinear  (efficient for sparse, high-dim features)
    - class_weight='balanced' to handle class imbalance
    - max_iter=2000 to ensure convergence

    Parameters
    ----------
    random_state : int
        Controls reproducibility of the LR solver.

    Returns
    -------
    sklearn.pipeline.Pipeline
    """
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=20_000,
            min_df=2,
            max_df=0.95,
            sublinear_tf=True,
            lowercase=True,
        )),
        ("lr", LogisticRegression(
            max_iter=2_000,
            class_weight="balanced",
            solver="liblinear",
            random_state=random_state,
        )),
    ])

---
## 5 · Evaluation Helpers

In [ ]:
def run_cv(
    df: pd.DataFrame,
    name: str,
    cv_random_state: int = RANDOM_STATE,
    lr_random_state: int = RANDOM_STATE,
) -> tuple[float, float, float, float]:
    """
    Run 10-fold stratified CV and report F1 + Accuracy.

    Returns
    -------
    (f1_mean, f1_std, acc_mean, acc_std)
    """
    X, y = df["text"], df["label"]
    pipe = build_pipeline(random_state=lr_random_state)
    cv   = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=cv_random_state)

    f1_scores  = cross_val_score(pipe, X, y, cv=cv, scoring="f1")
    acc_scores = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy")

    print(f"\n{name}  [{N_SPLITS}-fold CV]")
    print(f"  F1   mean={f1_scores.mean():.4f}  std={f1_scores.std():.4f}")
    print(f"  ACC  mean={acc_scores.mean():.4f}  std={acc_scores.std():.4f}")
    return f1_scores.mean(), f1_scores.std(), acc_scores.mean(), acc_scores.std()


def evaluate_model(
    model: Pipeline,
    X: pd.Series,
    y: pd.Series,
    title: str,
) -> tuple[float, float, np.ndarray]:
    """
    Evaluate a fitted pipeline and print a classification report.

    Returns
    -------
    (accuracy, f1, predictions)
    """
    pred = model.predict(X)
    acc  = accuracy_score(y, pred)
    f1   = f1_score(y, pred)

    print(f"\n{title}")
    print(f"  Accuracy: {acc:.4f}  |  F1: {f1:.4f}")
    print(classification_report(y, pred, digits=4))
    print("Confusion matrix:\n", confusion_matrix(y, pred))
    return acc, f1, pred

---
## 6 · Monolingual Cross-Fold Evaluation

For each fold of the source language:
1. Train on the fold's training split.
2. Evaluate **in-domain** on the fold's held-out split.
3. Evaluate **cross-lingual zero-shot** on the full target-language test set.
4. Evaluate on the **combined** test set.

In [ ]:
def run_monolingual_model_cv(
    df_source: pd.DataFrame,
    df_target_test: pd.DataFrame,
    df_combined_test: pd.DataFrame,
    name: str,
    cv_random_state: int = RANDOM_STATE,
    lr_random_state: int = RANDOM_STATE,
) -> tuple:
    """
    Extended CV for a monolingual model, evaluating across three test conditions.

    Parameters
    ----------
    df_source        : Full source-language dataset (CV is applied here).
    df_target_test   : Full target-language test set (cross-lingual zero-shot).
    df_combined_test : Combined bilingual test set.
    name             : Label for printed output.

    Returns
    -------
    12-tuple of (mean, std) pairs for:
        in-domain acc/f1, cross-lingual acc/f1, combined acc/f1
    """
    X_src, y_src = df_source["text"], df_source["label"]
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=cv_random_state)

    in_dom_accs, in_dom_f1s   = [], []
    cross_accs,  cross_f1s    = [], []
    comb_accs,   comb_f1s     = [], []

    print(f"\n--- Monolingual Extended CV: {name} ---")
    for fold, (train_idx, test_idx) in enumerate(cv.split(X_src, y_src)):
        print(f"  Fold {fold + 1}/{N_SPLITS}")

        pipe = build_pipeline(random_state=lr_random_state)
        pipe.fit(X_src.iloc[train_idx], y_src.iloc[train_idx])

        acc, f1, _ = evaluate_model(pipe, X_src.iloc[test_idx], y_src.iloc[test_idx],
                                    f"    Fold {fold+1} {name} (in-domain)")
        in_dom_accs.append(acc); in_dom_f1s.append(f1)

        acc, f1, _ = evaluate_model(pipe, df_target_test["text"], df_target_test["label"],
                                    f"    Fold {fold+1} {name} (cross-lingual zero-shot)")
        cross_accs.append(acc); cross_f1s.append(f1)

        acc, f1, _ = evaluate_model(pipe, df_combined_test["text"], df_combined_test["label"],
                                    f"    Fold {fold+1} {name} (combined test)")
        comb_accs.append(acc); comb_f1s.append(f1)

    print(f"\n  Summary: {name}")
    print(f"  In-domain      F1={np.mean(in_dom_f1s):.4f}±{np.std(in_dom_f1s):.4f}  "
          f"ACC={np.mean(in_dom_accs):.4f}±{np.std(in_dom_accs):.4f}")
    print(f"  Cross-lingual  F1={np.mean(cross_f1s):.4f}±{np.std(cross_f1s):.4f}  "
          f"ACC={np.mean(cross_accs):.4f}±{np.std(cross_accs):.4f}")
    print(f"  Combined       F1={np.mean(comb_f1s):.4f}±{np.std(comb_f1s):.4f}  "
          f"ACC={np.mean(comb_accs):.4f}±{np.std(comb_accs):.4f}")

    return (
        np.mean(in_dom_accs),  np.std(in_dom_accs),  np.mean(in_dom_f1s),  np.std(in_dom_f1s),
        np.mean(cross_accs),   np.std(cross_accs),   np.mean(cross_f1s),   np.std(cross_f1s),
        np.mean(comb_accs),    np.std(comb_accs),    np.mean(comb_f1s),    np.std(comb_f1s),
    )

In [ ]:
(
    en_id_acc, en_id_acc_std, en_id_f1, en_id_f1_std,
    en_cl_acc, en_cl_acc_std, en_cl_f1, en_cl_f1_std,
    en_comb_acc, en_comb_acc_std, en_comb_f1, en_comb_f1_std,
) = run_monolingual_model_cv(df_en, df_tl, test_df, "English-trained model")

In [ ]:
(
    tl_id_acc, tl_id_acc_std, tl_id_f1, tl_id_f1_std,
    tl_cl_acc, tl_cl_acc_std, tl_cl_f1, tl_cl_f1_std,
    tl_comb_acc, tl_comb_acc_std, tl_comb_f1, tl_comb_f1_std,
) = run_monolingual_model_cv(df_tl, df_en, test_df, "Tagalog-trained model")

---
## 7 · Bilingual Cross-Fold Evaluation

CV is applied to the combined training set; the fixed held-out test sets are reused across all folds to allow per-language fairness checks.

In [ ]:
def run_bilingual_model_extended_cv(
    df_train_pool: pd.DataFrame,
    df_overall_test: pd.DataFrame,
    df_en_test: pd.DataFrame,
    df_tl_test: pd.DataFrame,
    name: str,
    cv_random_state: int = RANDOM_STATE,
    lr_random_state: int = RANDOM_STATE,
) -> tuple:
    """
    Extended CV for the bilingual model with per-language fairness checks.

    Note: Only the training index from each CV fold is used; evaluation is always
    done on the fixed held-out test sets (not the fold's validation split).

    Returns
    -------
    12-tuple of (mean, std) for overall, English, and Tagalog test performance.
    """
    X_pool, y_pool = df_train_pool["text"], df_train_pool["label"]
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=cv_random_state)

    overall_accs, overall_f1s = [], []
    en_accs, en_f1s           = [], []
    tl_accs, tl_f1s           = [], []

    print(f"\n--- Bilingual Extended CV: {name} ---")
    for fold, (train_idx, _) in enumerate(cv.split(X_pool, y_pool)):
        print(f"  Fold {fold + 1}/{N_SPLITS}")

        pipe = build_pipeline(random_state=lr_random_state)
        pipe.fit(X_pool.iloc[train_idx], y_pool.iloc[train_idx])

        acc, f1, _ = evaluate_model(pipe, df_overall_test["text"], df_overall_test["label"],
                                    f"    Fold {fold+1} {name} (overall test)")
        overall_accs.append(acc); overall_f1s.append(f1)

        acc, f1, _ = evaluate_model(pipe, df_en_test["text"], df_en_test["label"],
                                    f"    Fold {fold+1} {name} (English fairness check)")
        en_accs.append(acc); en_f1s.append(f1)

        acc, f1, _ = evaluate_model(pipe, df_tl_test["text"], df_tl_test["label"],
                                    f"    Fold {fold+1} {name} (Tagalog fairness check)")
        tl_accs.append(acc); tl_f1s.append(f1)

    print(f"\n  Summary: {name}")
    print(f"  Overall   F1={np.mean(overall_f1s):.4f}±{np.std(overall_f1s):.4f}  "
          f"ACC={np.mean(overall_accs):.4f}±{np.std(overall_accs):.4f}")
    print(f"  English   F1={np.mean(en_f1s):.4f}±{np.std(en_f1s):.4f}  "
          f"ACC={np.mean(en_accs):.4f}±{np.std(en_accs):.4f}")
    print(f"  Tagalog   F1={np.mean(tl_f1s):.4f}±{np.std(tl_f1s):.4f}  "
          f"ACC={np.mean(tl_accs):.4f}±{np.std(tl_accs):.4f}")

    return (
        np.mean(overall_accs), np.std(overall_accs), np.mean(overall_f1s), np.std(overall_f1s),
        np.mean(en_accs),      np.std(en_accs),      np.mean(en_f1s),      np.std(en_f1s),
        np.mean(tl_accs),      np.std(tl_accs),      np.mean(tl_f1s),      np.std(tl_f1s),
    )

---
## 8 · Monolingual Full-Dataset CV (Baselines)

In [ ]:
en_cv_f1_mean, en_cv_f1_std, en_cv_acc_mean, en_cv_acc_std = run_cv(df_en, "English — full dataset CV")
tl_cv_f1_mean, tl_cv_f1_std, tl_cv_acc_mean, tl_cv_acc_std = run_cv(df_tl, "Tagalog — full dataset CV")

In [ ]:
bi_cv_f1_mean, bi_cv_f1_std, bi_cv_acc_mean, bi_cv_acc_std = run_cv(
    df_all, "Bilingual — full combined dataset CV (baseline)"
)

---
## 9 · Extended CV — All Conditions

In [ ]:
(
    en_id_acc, en_id_acc_std, en_id_f1, en_id_f1_std,
    en_cl_acc, en_cl_acc_std, en_cl_f1, en_cl_f1_std,
    en_comb_acc, en_comb_acc_std, en_comb_f1, en_comb_f1_std,
) = run_monolingual_model_cv(df_en, df_tl, test_df, "English-trained model")

In [ ]:
(
    tl_id_acc, tl_id_acc_std, tl_id_f1, tl_id_f1_std,
    tl_cl_acc, tl_cl_acc_std, tl_cl_f1, tl_cl_f1_std,
    tl_comb_acc, tl_comb_acc_std, tl_comb_f1, tl_comb_f1_std,
) = run_monolingual_model_cv(df_tl, df_en, test_df, "Tagalog-trained model")

In [ ]:
(
    multi_overall_acc, multi_overall_acc_std, multi_overall_f1, multi_overall_f1_std,
    multi_en_acc, multi_en_acc_std, multi_en_f1, multi_en_f1_std,
    multi_tl_acc, multi_tl_acc_std, multi_tl_f1, multi_tl_f1_std,
) = run_bilingual_model_extended_cv(
    train_df, test_df, en_test, tl_test, "Bilingual model"
)

---
## 10 · Held-Out Test Evaluation (Monolingual Models)

In [ ]:
# English model
en_model = build_pipeline()
en_model.fit(en_train["text"], en_train["label"])

en_internal_acc, en_internal_f1, en_internal_pred = evaluate_model(
    en_model, en_test["text"], en_test["label"],
    "English model → English test (in-domain)"
)
en_zeroshot_acc, en_zeroshot_f1, en_zeroshot_pred = evaluate_model(
    en_model, tl_test["text"], tl_test["label"],
    "English model → Tagalog test (cross-lingual zero-shot)"
)
en_combined_acc, en_combined_f1, en_combined_pred = evaluate_model(
    en_model, test_df["text"], test_df["label"],
    "English model → Combined test"
)

In [ ]:
# Tagalog model
tl_model = build_pipeline()
tl_model.fit(tl_train["text"], tl_train["label"])

tl_internal_acc, tl_internal_f1, tl_internal_pred = evaluate_model(
    tl_model, tl_test["text"], tl_test["label"],
    "Tagalog model → Tagalog test (in-domain)"
)
tl_zeroshot_acc, tl_zeroshot_f1, tl_zeroshot_pred = evaluate_model(
    tl_model, en_test["text"], en_test["label"],
    "Tagalog model → English test (cross-lingual zero-shot)"
)
tl_combined_acc, tl_combined_f1, tl_combined_pred = evaluate_model(
    tl_model, test_df["text"], test_df["label"],
    "Tagalog model → Combined test"
)

---
## 11 · Results Summary Table

In [ ]:
rows = [
    # English-only model
    ["English-only", "English (in-domain CV)",         en_id_acc,       en_id_f1,       en_id_acc_std,       en_id_f1_std],
    ["English-only", "Tagalog (cross-lingual CV)",     en_cl_acc,       en_cl_f1,       en_cl_acc_std,       en_cl_f1_std],
    ["English-only", "Combined (mixed-domain CV)",     en_comb_acc,     en_comb_f1,     en_comb_acc_std,     en_comb_f1_std],
    ["English-only", "Full dataset CV (baseline)",     en_cv_acc_mean,  en_cv_f1_mean,  en_cv_acc_std,       en_cv_f1_std],
    # Tagalog-only model
    ["Tagalog-only", "Tagalog (in-domain CV)",         tl_id_acc,       tl_id_f1,       tl_id_acc_std,       tl_id_f1_std],
    ["Tagalog-only", "English (cross-lingual CV)",     tl_cl_acc,       tl_cl_f1,       tl_cl_acc_std,       tl_cl_f1_std],
    ["Tagalog-only", "Combined (mixed-domain CV)",     tl_comb_acc,     tl_comb_f1,     tl_comb_acc_std,     tl_comb_f1_std],
    ["Tagalog-only", "Full dataset CV (baseline)",     tl_cv_acc_mean,  tl_cv_f1_mean,  tl_cv_acc_std,       tl_cv_f1_std],
    # Bilingual model
    ["Bilingual", "Full dataset CV (baseline)",     bi_cv_acc_mean,  bi_cv_f1_mean,  bi_cv_acc_std,       bi_cv_f1_std],
    ["Bilingual", "Combined (overall CV)",          multi_overall_acc, multi_overall_f1, multi_overall_acc_std, multi_overall_f1_std],
    ["Bilingual", "English (fairness check CV)",   multi_en_acc,    multi_en_f1,    multi_en_acc_std,    multi_en_f1_std],
    ["Bilingual", "Tagalog (fairness check CV)",   multi_tl_acc,    multi_tl_f1,    multi_tl_acc_std,    multi_tl_f1_std],
]

results = pd.DataFrame(rows, columns=["Model", "Evaluation", "Accuracy", "F1", "Accuracy_std", "F1_std"])
results[["Accuracy", "F1", "Accuracy_std", "F1_std"]] = results[["Accuracy", "F1", "Accuracy_std", "F1_std"]].round(4)

print(results.to_string(index=False))